In [1]:
import torch
from transformer import make_causal_mask, scaled_dot_product_attention

T = 4
d = 8
batch = 2
heads = 2

torch.manual_seed(0)
Q = torch.randn(batch, heads, T, d)
K = torch.randn(batch, heads, T, d)
V = torch.randn(batch, heads, T, d)
mask = make_causal_mask(time_steps=T, device=Q.device)

output, weights = scaled_dot_product_attention(
    query=Q, key=K, value=V, mask=mask
)

print("mask\n", mask[0, 0])
print("weights row 0\n", weights[0, 0, 0])
print("output", tuple(output.shape))

assert output.shape == (batch, heads, T, d)
assert weights.shape == (batch, heads, T, T)
# first token may only look at itself
assert torch.allclose(weights[:, :, 0, 1:], torch.zeros_like(weights[:, :, 0, 1:]), atol=1e-6)
# no NaNs
assert torch.isfinite(output).all()
print("attention test ok")

mask
 tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])
weights row 0
 tensor([1., 0., 0., 0.])
output (2, 2, 4, 8)
attention test ok


In [2]:
import importlib
import transformer
importlib.reload(transformer)
from transformer import MultiHeadSelfAttention

batch, T, d_model, n_heads = 2, 8, 64, 4
attn = MultiHeadSelfAttention(d_model=d_model, n_heads=n_heads, dropout=0.0)
x = torch.randn(batch, T, d_model)
y = attn.forward(x=x)

print("x", tuple(x.shape))
print("y", tuple(y.shape))
assert y.shape == x.shape
assert torch.isfinite(y).all()
print("multi-head test ok")

x (2, 8, 64)
y (2, 8, 64)
multi-head test ok


In [3]:
import importlib
import transformer
importlib.reload(transformer)
from transformer import FeedForward, TransformerBlock

batch, T, d_model, n_heads, d_ff = 2, 8, 64, 4, 256
x = torch.randn(batch, T, d_model)

ffn = FeedForward(d_model=d_model, d_ff=d_ff, dropout=0.0)
y_ffn = ffn.forward(x=x)
print("ffn", tuple(y_ffn.shape))
assert y_ffn.shape == x.shape

block = TransformerBlock(
    d_model=d_model, n_heads=n_heads, d_ff=d_ff, dropout=0.0
)
y = block.forward(x=x)
print("block", tuple(y.shape))
assert y.shape == x.shape
assert torch.isfinite(y).all()
print("block test ok")

ffn (2, 8, 64)
block (2, 8, 64)
block test ok


In [5]:
import importlib
import transformer
importlib.reload(transformer)
from transformer import TransformerBackbone

batch, T, d_model = 2, 16, 64
backbone = TransformerBackbone(
    d_model=d_model,
    n_heads=4,
    d_ff=256,
    n_layers=2,
    max_len=32,
    dropout=0.0,
)
x = torch.randn(batch, T, d_model)
output, extra = backbone.forward(x=x, hidden=None)

print("x     ", tuple(x.shape))
print("output", tuple(output.shape))
print("extra ", extra)
assert output.shape == (batch, T, d_model)
assert extra is None
assert torch.isfinite(output).all()
print("backbone test ok")

x      (2, 16, 64)
output (2, 16, 64)
extra  None
backbone test ok


In [6]:
import torch
from model import NextWordModel

batch, T, vocab_size = 2, 8, 50
net = NextWordModel(
    vocab_size=vocab_size,
    embed_size=64,
    n_heads=4,
    d_ff=256,
    n_layers=2,
    max_len=32,
    pad_id=0,
    dropout=0.0,
)
net.eval()

input_ids = torch.randint(1, vocab_size, (batch, T))
logits = net.forward(input_ids=input_ids)
print("input_ids", tuple(input_ids.shape))
print("logits   ", tuple(logits.shape))
assert logits.shape == (batch, T, vocab_size)
print("Q2 NextWordModel test ok")

input_ids (2, 8)
logits    (2, 8, 50)
Q2 NextWordModel test ok


In [7]:
import sys
from pathlib import Path
import torch

sys.path.append(str(Path("..").resolve() / "question_one"))
from dataset import build_dataloaders
from model import NextWordModel
from utils import compute_loss

# dataset.py lives in question_one; run this notebook from question_two
sys.path.append(str(Path("..").resolve() / "question_one"))
from dataset import build_dataloaders as q1_data

loaders, vocab, stats = q1_data(batch_size=4, min_freq=2)
batch = next(iter(loaders["train"]))

net = NextWordModel(
    vocab_size=stats["vocab_size"],
    embed_size=64,
    pad_id=vocab.pad_id(),
    dropout=0.0,
)
logits = net.forward(input_ids=batch["input_ids"])
loss = compute_loss(logits=logits, labels=batch["labels"], pad_id=vocab.pad_id())
print("logits", tuple(logits.shape), "loss", float(loss))
print("dataset + Q2 model test ok")

logits (4, 16, 5960) loss 8.873469352722168
dataset + Q2 model test ok


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14232\1712950606.py:25: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  print("logits", tuple(logits.shape), "loss", float(loss))
